# Tráfico en Madrid: Preparación del dataset

En este notebook preparamos los datos para entrenar los modelos. Partimos de los CSV mensuales del Ayuntamiento, que vienen comprimidos en ZIP y con calidad desigual, y los dejamos listos en ficheros parquet.

Primero vamos a limpiar el catálogo de sensores. Después procesamos el histórico mes a mes para no saturar la RAM. Al final dividimos el dataset en train, validación, test y una evaluación final separada con marzo de 2026, que reservamos para el notebook 04.

Las decisiones importantes, como los umbrales, los sensores que usamos y las features, vienen del notebook 01. 

In [1]:
%pip install -r ../requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


## 1. Configuración inicial

Vamos a dejar al principio todos los imports, constantes, rutas de archivos y configuración estética. Así lo tenemos todo en un único sitio para poder cambiarlo rápidamente.

In [2]:
from pathlib import Path
import gc
import zipfile

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split

Todas estas variables son las que elegimos en el notebook01. Además añadimos rangos y particiones de los datos.

In [3]:
RANDOM_STATE = 42

# Umbrales de congestión por tipo de sensor (justificados en NB01 sección 6.1)
UMBRAL_URB = 50
UMBRAL_M30 = 70
TIPO_ELEM_MAP = {"URB": 0, "M30": 1}
HORAS_PUNTA = [7, 8, 9, 14, 18, 19, 20]

# Rangos válidos para descartar centinelas antes del downcast
DISTRITO_MIN, DISTRITO_MAX = 1, 21
UTM_X_MIN, UTM_X_MAX = 400_000, 470_000
UTM_Y_MIN, UTM_Y_MAX = 4_460_000, 4_500_000

# Proporciones del split principal (80 / 10 / 10)
TEST_SIZE = 0.10
VAL_SIZE = TEST_SIZE / (1 - TEST_SIZE)  # 1/9 del bloque train+val

# Años para el dataset de entrenamiento (ampliable a [2024, 2025, 2026])
ANIOS_A_CARGAR = [2026]

# Mes más reciente — reservado para evaluación final
MES_VALIDACION_FINAL = "03-2026"
NOMBRE_EVAL_FINAL = "eval_final"

Ahora todas las rutas donde se van a leer y guardar los archivos.

In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_RAW = Path("../data/raw")
DATA_HIST_ZIPS = DATA_RAW / "historico"
DATA_SENS_RAW = DATA_RAW / "sensores" / "sensores.csv"

DATA_PROC = Path("../data/processed")
DATA_HIST_CSV = DATA_RAW / "historico_csv"
DATA_HIST_PARQUET = DATA_PROC / "historico"
DATA_SENS_DIR = DATA_PROC / "sensores"
DATA_SENS_OUT = DATA_SENS_DIR / "sensores.parquet"

DATA_FINAL = Path("../data/final")

for carpeta in (DATA_PROC, DATA_HIST_CSV, DATA_HIST_PARQUET, DATA_SENS_DIR, DATA_FINAL):
    carpeta.mkdir(parents=True, exist_ok=True)

print("Entorno configurado.")

Entorno configurado.


## 2. Procesamiento del catálogo de sensores

El catálogo de sensores es un CSV es pequeño. Como en el notebook 01 vimos que los sensores C30 no tienen carga, los quitamos. Después hacemos una limpieza simple: eliminamos duplicados, nulos, valores fuera de rango y ajustamos tipos de datos para ahorrar memoria.

In [5]:
def limpiar_sensores() -> pd.DataFrame:
    """Limpia el CSV de sensores. Excluye C30 (sin variable carga, ver NB01 sección 5.3)."""
    df = pd.read_csv(DATA_SENS_RAW, sep=";", encoding="latin-1")

    cols = ["tipo_elem", "distrito", "id", "nombre", "utm_x", "utm_y"]
    df = df[cols]

    df = df.replace(r"^\s*$", np.nan, regex=True)

    n_prev = len(df)
    df = df.drop_duplicates()
    print(f"Sensores: duplicados eliminados = {n_prev - len(df):,}")

    n_prev = len(df)
    df = df.dropna()
    print(f"Sensores: nulos/vacíos eliminados = {n_prev - len(df):,}")

    n_prev = len(df)
    df = df[df["tipo_elem"] != "C30"]
    print(f"Sensores: C30 eliminados = {n_prev - len(df):,}")

    n_prev = len(df)
    df = df[df["tipo_elem"].isin(TIPO_ELEM_MAP)]
    print(f"Sensores: tipo desconocido elim. = {n_prev - len(df):,}")

    return df

Validamos los rangos antes de cambiar los tipos de datos. Así evitamos que entren valores fuera de sitio, como un distrito que no existe o un id inválido. Por otra parte fijamos los rangos de UTM para que estén dentro de Madrid.

In [6]:
def validar_rangos_sensores(df: pd.DataFrame) -> pd.DataFrame:
    """Descarta filas con datos invalidos o valores fuera de los rangos esperables."""
    filtros = {
        f"distrito in [{DISTRITO_MIN}, {DISTRITO_MAX}]": df["distrito"].between(
            DISTRITO_MIN, DISTRITO_MAX
        ),
        "id > 0": df["id"] > 0,
        f"utm_x in [{UTM_X_MIN/1000:.0f}k, {UTM_X_MAX/1000:.0f}k]": df["utm_x"].between(
            UTM_X_MIN, UTM_X_MAX
        ),
        f"utm_y in [{UTM_Y_MIN/1000:.0f}k, {UTM_Y_MAX/1000:.0f}k]": df["utm_y"].between(
            UTM_Y_MIN, UTM_Y_MAX
        ),
    }
    for nombre, mascara in filtros.items():
        n_prev = len(df)
        df = df[mascara]
        fuera = n_prev - len(df)
        if fuera:
            print(f"Sensores: fuera de rango '{nombre}' = {fuera:,}")
    return df

Con el dataset ya limpio, pasamos cada columna al tipo de dato más ligero posible.

In [7]:
def preparar_tipos_sensores(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["tipo_elem"] = df["tipo_elem"].map(TIPO_ELEM_MAP).astype("int8")
    df["distrito"] = df["distrito"].astype("int8")
    df["id"] = df["id"].astype("int32")
    df["nombre"] = df["nombre"].astype("string")
    df["utm_x"] = df["utm_x"].astype("float32")
    df["utm_y"] = df["utm_y"].astype("float32")
    return df

Ejecutamos la limpieza de sensores

In [8]:
if not DATA_SENS_OUT.exists():
    print("Generando catálogo de sensores desde CSV...")
    df_sens = limpiar_sensores()
    df_sens = validar_rangos_sensores(df_sens)
    df_sens = preparar_tipos_sensores(df_sens)
    df_sens.to_parquet(DATA_SENS_OUT, compression="snappy", index=False)
    print(f"Guardado: {DATA_SENS_OUT.name} ({len(df_sens):,} sensores)")
    del df_sens
    gc.collect()
else:
    print(f"Ya existía {DATA_SENS_OUT.name}, se omite la limpieza.")

Generando catálogo de sensores desde CSV...
Sensores: duplicados eliminados = 0
Sensores: nulos/vacíos eliminados = 11
Sensores: C30 eliminados = 0
Sensores: tipo desconocido elim. = 127
Guardado: sensores.parquet (4,824 sensores)


Cargamos siempre desde disco para empezar desde el mismo punto. Después de la limpieza, nos quedamos con 4.824 sensores en vez de 4.962, porque quitamos C30, duplicados y filas fuera de rango.

In [9]:
df_sensores = pd.read_parquet(DATA_SENS_OUT)

memoria_kb = df_sensores.memory_usage(deep=True).sum() / 1024
print(f"Catálogo cargado: {len(df_sensores):,} sensores, {df_sensores.shape[1]} columnas")
print(f"Memoria: {memoria_kb:.1f} KB")
df_sensores.head()

Catálogo cargado: 4,824 sensores, 6 columnas
Memoria: 317.5 KB


,tipo_elem,distrito,id,nombre,utm_x,utm_y
0,0,8,5902,(TACTICO) ISLAS CIES O-E,439227.56,4481434.00
1,0,11,5088,ARLANZA E-O,439202.94,4470942.00
2,0,11,5141,Cº VIEJO DE LEGANES N-S,437756.78,4470016.50
3,0,10,5010,MARQUES MONISTROL N-S (CALLE 30),438563.16,4474058.00
4,0,5,3924,Doctor Arce S-N - Estevez-Pl. Cataluña,442320.03,4477674.50


## 3. Procesamiento del histórico de tráfico

El histórico viene en ZIPs mensuales y tiene muchísimas filas, así que lo procesamos mes a mes para no llenar la RAM. Cada mes se limpia, se transforma y se guarda en parquet.

Los archivos intermedios se guardan en `processed/historico/`

In [10]:
zips = sorted(DATA_HIST_ZIPS.glob("*.zip"))
print(f"ZIPs encontrados: {len(zips)}")

for zip_path in zips:
    with zipfile.ZipFile(zip_path) as z:
        nombre_csv = z.namelist()[0]
        csv_path = DATA_HIST_CSV / nombre_csv
        if csv_path.exists():
            print(f"  {zip_path.name}: ya extraído, se omite")
            continue
        z.extractall(DATA_HIST_CSV)
    print(f"  {zip_path.name}: extraído -> {nombre_csv}")

csvs = sorted(DATA_HIST_CSV.glob("*.csv"))
print(f"CSVs listos para procesar: {len(csvs)}")

ZIPs encontrados: 27
  2024-01.zip: ya extraído, se omite
  2024-02.zip: ya extraído, se omite
  2024-03.zip: ya extraído, se omite
  2024-04.zip: ya extraído, se omite
  2024-05.zip: ya extraído, se omite
  2024-06.zip: ya extraído, se omite
  2024-07.zip: ya extraído, se omite
  2024-08.zip: ya extraído, se omite
  2024-09.zip: ya extraído, se omite
  2024-10.zip: ya extraído, se omite
  2024-11.zip: ya extraído, se omite
  2024-12.zip: ya extraído, se omite
  2025-01.zip: ya extraído, se omite
  2025-02.zip: ya extraído, se omite
  2025-03.zip: ya extraído, se omite
  2025-04.zip: ya extraído, se omite
  2025-05.zip: ya extraído, se omite
  2025-06.zip: ya extraído, se omite
  2025-07.zip: ya extraído, se omite
  2025-08.zip: ya extraído, se omite
  2025-09.zip: ya extraído, se omite
  2025-10.zip: ya extraído, se omite
  2025-11.zip: ya extraído, se omite
  2025-12.zip: ya extraído, se omite
  2026-01.zip: ya extraído, se omite
  2026-02.zip: ya extraído, se omite
  2026-03.zip: ya

`procesar_mes` limpia cada mes. Primero quita las filas erróneas y las que tienen valores raros en carga. Luego une cada lectura con los datos del sensor para añadir tipo, distrito y coordenadas.

Después agrupamos los datos por hora usando el máximo de carga de ese intervalo. Así reducimos el volumen sin perder los picos de congestión, que son justo lo que nos interesa detectar. Si usáramos la media podríamos perder algunos atascos cortos. Si no agrupamos tenemos demasiados datos y mi portatil no procesa bien los datos.

`_generar_features` trabaja ya con esos datos por hora. Saca hora y dia_semana de fecha, crea las variables es_laborable y es_hora_punta, y calcula congestionado. Guardamos hora y dia_semana como enteros porque los modelos de árboles los usan directamente, y luego en el notebook 03 haremos la transformación sin/cos para la regresión logística.

In [11]:
def _generar_features(df: pd.DataFrame) -> pd.DataFrame:
    """Añade features temporales y el target binario; elimina la columna fecha."""
    hora = df["fecha"].dt.hour
    dia_sem = df["fecha"].dt.dayofweek

    df = df.copy()
    df["hora"] = hora.astype("int8")
    df["dia_semana"] = dia_sem.astype("int8")
    df["es_laborable"] = (dia_sem <= 4).astype("bool")
    df["es_hora_punta"] = hora.isin(HORAS_PUNTA).astype("bool")

    umbral = np.where(df["tipo_elem"] == TIPO_ELEM_MAP["URB"], UMBRAL_URB, UMBRAL_M30)
    df["congestionado"] = (df["carga"] >= umbral).astype("bool")

    df = df.drop(columns=["fecha"])
    return df


def procesar_mes(csv_path: Path, sensores: pd.DataFrame) -> None:
    """Pipeline completo para un mes: lectura, limpieza, agregación horaria, features y guardado."""
    mes = csv_path.stem
    destino = DATA_HIST_PARQUET / f"{mes}.parquet"
    if destino.exists():
        print(f"[{mes}]: ya procesado, se omite")
        return

    df = pd.read_csv(
        csv_path,
        sep=";",
        usecols=["id", "fecha", "carga", "error"],
        parse_dates=["fecha"],
        encoding="latin-1",
        on_bad_lines="skip",
        engine="pyarrow",
    )
    print(f"[{mes}]")
    print(f"leídas {len(df)}")

    n_prev = len(df)
    df = df[df["error"] == "N"]
    print(f"error sensor {n_prev - len(df)}")

    n_prev = len(df)
    df = df.dropna()
    print(f"nulos {n_prev - len(df)}")

    n_prev = len(df)
    df = df[df["carga"].between(0, 100)]
    df = df[df["id"] > 0]
    print(f"fuera rango {n_prev - len(df)}")

    df = df.merge(sensores, on="id", how="inner")
    print(f"tras merge {len(df)}")

    # Juntar por hora
    df = df.drop(columns=["error", "nombre"])
    df["fecha"] = df["fecha"].dt.floor("h")
    n_prev = len(df)
    df = (
        df.groupby(["id", "fecha"], sort=False)
        .agg(
            carga=("carga", "max"),
            tipo_elem=("tipo_elem", "first"),
            distrito=("distrito", "first"),
            utm_x=("utm_x", "first"),
            utm_y=("utm_y", "first"),
        )
        .reset_index()
    )
    print(f"tras agreg. {len(df)}  (-{n_prev - len(df)} filas)")

    df = _generar_features(df)

    df["id"] = df["id"].astype("int32")
    df["distrito"] = df["distrito"].astype("int8")
    df["tipo_elem"] = df["tipo_elem"].astype("int8")
    df["carga"] = df["carga"].astype("float32")
    df["utm_x"] = df["utm_x"].astype("float32")
    df["utm_y"] = df["utm_y"].astype("float32")

    df.to_parquet(destino, compression="snappy", index=False)
    mb = df.memory_usage(deep=True).sum() / 1024**2
    print(f"guardadas {len(df)}  ({mb:.1f} MB en memoria)")

Procesamos mes a mes.

In [12]:
for csv_path in csvs:
    procesar_mes(csv_path, df_sensores)
    gc.collect()

print(f"Meses procesados: {len(csvs)}")

[01-2024]
leídas 13343452
error sensor 22017
nulos 0
fuera rango 0
tras merge 12913178
tras agreg. 3280348  (-9632830 filas)
guardadas 3280348  (72.0 MB en memoria)
[01-2025]
leídas 13218804
error sensor 19891
nulos 0
fuera rango 0
tras merge 12835222
tras agreg. 3265982  (-9569240 filas)
guardadas 3265982  (71.6 MB en memoria)
[01-2026]
leídas 14296853
error sensor 0
nulos 0
fuera rango 0
tras merge 13767363
tras agreg. 3445452  (-10321911 filas)
guardadas 3445452  (75.6 MB en memoria)
[02-2024]
leídas 12532935
error sensor 19097
nulos 0
fuera rango 0
tras merge 12135385
tras agreg. 3087044  (-9048341 filas)
guardadas 3087044  (67.7 MB en memoria)
[02-2025]
leídas 12056165
error sensor 27135
nulos 0
fuera rango 0
tras merge 11707275
tras agreg. 2969727  (-8737548 filas)
guardadas 2969727  (65.1 MB en memoria)
[02-2026]
leídas 11948889
error sensor 56032
nulos 0
fuera rango 0
tras merge 11424037
tras agreg. 2880568  (-8543469 filas)
guardadas 2880568  (63.2 MB en memoria)
[03-2024]
leí

## 4. Consolidación del dataset final

Con todos los meses ya procesados, separamos el mes más reciente para la evaluación final. En este caso guardamos marzo de 2026 aparte para simular una predicción real, sin que los modelos lo vean hasta el notebook 04.

El resto de meses forman el dataset principal. No los cargamos todos a la vez: solo guardamos sus rutas y los procesamos uno a uno en la sección 7.

In [13]:
parquet_files = sorted(DATA_HIST_PARQUET.glob("*.parquet"))

mes_eval_final_path = next(p for p in parquet_files if p.stem == MES_VALIDACION_FINAL)
df_eval_final = pd.read_parquet(mes_eval_final_path)

anios_objetivo = tuple(str(anio) for anio in ANIOS_A_CARGAR)
parquets_main = [
    p
    for p in parquet_files
    if p.stem != MES_VALIDACION_FINAL and any(anio in p.stem for anio in anios_objetivo)
]

print(f"Evaluación final: {mes_eval_final_path.stem} son {len(df_eval_final):,} filas")
print(f"Años para dataset principal: {ANIOS_A_CARGAR}")
print(f"Meses usados en main: {len(parquets_main)}")
print(f"Primer mes main: {parquets_main[0].stem} | Último mes main: {parquets_main[-1].stem}")

Evaluación final: 03-2026 son 3,246,364 filas
Años para dataset principal: [2026]
Meses usados en main: 2
Primer mes main: 01-2026 | Último mes main: 02-2026


## 5. División de los datos (train / val / test)

Hacemos el split mes a mes para no cargar todo el dataset en memoria. Además, lo estratificamos con congestionado para que train, val y test tengan una proporción parecida de positivos.

In [14]:
def split_indices_estratificados(y_mes_np: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Devuelve indices train/val/test para un mes."""
    indices_mes = np.arange(len(y_mes_np), dtype=np.int32)

    # Primer split: test contra train+val
    indices_train_val, indices_test = train_test_split(
        indices_mes,
        test_size=TEST_SIZE,
        stratify=y_mes_np,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    # Segundo split: train contra val dentro del bloque train+val
    indices_train, indices_val = train_test_split(
        indices_train_val,
        test_size=VAL_SIZE,
        stratify=y_mes_np[indices_train_val],
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    return indices_train, indices_val, indices_test

## 6. Guardado final

Como el dataset principal es muy grande, lo procesamos mes a mes para no gastar demasiada RAM. Cada mes se divide en train, val y test y se guarda directamente en disco.

Se ha intentado hacer de manera normal cargando todo en memoria primero y luego partiendolo pero como son tantos datos no tengo suficiente RAM para todo. Así para escribirlo usamos ParquetWriter, que permite ir añadiendo datos sin tener que juntar todo antes. Así hacemos lo mismo con mucho menos uso de memoria.

In [15]:
TARGET_COL = "congestionado"
DROP_COLS = [TARGET_COL, "carga"]
FEATURE_COLS = [col for col in df_eval_final.columns if col not in DROP_COLS]
SPLITS_PRINCIPALES = ("train", "val", "test")


def append_df_to_parquet(
    writer: pq.ParquetWriter | None,
    df_chunk: pd.DataFrame,
    destino: Path,
) -> pq.ParquetWriter:
    """Escribe un chunk en parquet. Crea el writer la primera vez que se llama."""
    tabla = pa.Table.from_pandas(df_chunk, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(destino, tabla.schema, compression="snappy")
    writer.write_table(tabla)
    return writer


def num_rows_parquet(path: Path) -> int:
    """Devuelve el número de filas de un parquet sin cargarlo en memoria."""
    return pq.ParquetFile(path).metadata.num_rows

Antes de ejecutar el proceso, cerramos writers abiertos y borramos salidas anteriores para empezar limpio.
Así, si falla a mitad, podemos relanzar la celda sin dejar ficheros corruptos ni inconsistentes.

Sin esto se traba al intentar reejecutar en notebook

In [16]:
def cerrar_writers_previos_si_existen() -> None:
    """Cierra writers que hayan quedado abiertos de una ejecución interrumpida."""
    for nombre_var in ("writers_x", "writers_y"):
        writers_previos = globals().get(nombre_var, {})
        if isinstance(writers_previos, dict):
            for writer in writers_previos.values():
                if writer is not None:
                    writer.close()
    gc.collect()


def borrar_salidas_previas() -> None:
    """Elimina ficheros de salida para regenerarlos desde cero."""
    for nombre_split in (*SPLITS_PRINCIPALES, NOMBRE_EVAL_FINAL):
        for prefijo in ("X", "y"):
            salida = DATA_FINAL / f"{prefijo}_{nombre_split}.parquet"
            if salida.exists():
                salida.unlink()


def inicializar_acumuladores() -> tuple[dict, dict, dict, dict]:
    """Crea writers vacíos y contadores de filas y positivos por split."""
    writers_x = {split: None for split in SPLITS_PRINCIPALES}
    writers_y = {split: None for split in SPLITS_PRINCIPALES}
    filas_por_split = {split: 0 for split in SPLITS_PRINCIPALES}
    positivos_por_split = {split: 0 for split in SPLITS_PRINCIPALES}
    return writers_x, writers_y, filas_por_split, positivos_por_split


def cerrar_writers(writers_x: dict, writers_y: dict) -> None:
    """Cierra todos los writers una vez procesados todos los meses."""
    for writer in list(writers_x.values()) + list(writers_y.values()):
        if writer is not None:
            writer.close()

En cada mes hace la división en train, val y test, y guarda cada parte en su archivo parquet. La estratificación solo sirve para que el porcentaje de casos congestionados quede parecido en los tres conjuntos.

In [17]:
def guardar_chunk_split(
    df_split: pd.DataFrame,
    nombre_split: str,
    writers_x: dict,
    writers_y: dict,
) -> tuple[int, int]:
    """Guarda X e y de un split y devuelve (positivos, filas)."""
    x_split = df_split[FEATURE_COLS]
    y_split = pd.DataFrame({TARGET_COL: df_split[TARGET_COL]})
    writers_x[nombre_split] = append_df_to_parquet(
        writers_x[nombre_split], x_split, DATA_FINAL / f"X_{nombre_split}.parquet"
    )
    writers_y[nombre_split] = append_df_to_parquet(
        writers_y[nombre_split], y_split, DATA_FINAL / f"y_{nombre_split}.parquet"
    )
    positivos_chunk = int(y_split[TARGET_COL].sum())
    filas_chunk = len(df_split)
    del x_split, y_split
    return positivos_chunk, filas_chunk


def procesar_mes_para_splits(
    parquet_mes: Path,
    writers_x: dict,
    writers_y: dict,
    filas_por_split: dict,
    positivos_por_split: dict,
) -> None:
    """Lee un mes, lo divide en splits y escribe de forma incremental en los parquets finales."""
    df_mes = pd.read_parquet(parquet_mes)
    y_mes_np = df_mes[TARGET_COL].to_numpy(dtype=np.uint8, copy=False)
    indices_train, indices_val, indices_test = split_indices_estratificados(y_mes_np)
    indices_por_split = {"train": indices_train, "val": indices_val, "test": indices_test}
    print(f"[{parquet_mes.stem}] filas={len(df_mes):,}")
    for nombre_split, indices_split in indices_por_split.items():
        df_split = df_mes.iloc[indices_split]
        pos, filas = guardar_chunk_split(df_split, nombre_split, writers_x, writers_y)
        filas_por_split[nombre_split] += filas
        positivos_por_split[nombre_split] += pos
        del df_split
    del df_mes, y_mes_np, indices_train, indices_val, indices_test, indices_por_split
    gc.collect()


def guardar_eval_final() -> None:
    """Guarda el mes de evaluación final sin aplicar ningún split."""
    x_eval = df_eval_final[FEATURE_COLS]
    y_eval = pd.DataFrame({TARGET_COL: df_eval_final[TARGET_COL]})
    x_eval.to_parquet(
        DATA_FINAL / f"X_{NOMBRE_EVAL_FINAL}.parquet", compression="snappy", index=False
    )
    y_eval.to_parquet(
        DATA_FINAL / f"y_{NOMBRE_EVAL_FINAL}.parquet", compression="snappy", index=False
    )
    del x_eval, y_eval
    gc.collect()


def imprimir_resumen_distribucion(filas_por_split: dict, positivos_por_split: dict) -> int:
    """Imprime tamaño y tasa de positivos por split. Devuelve el total de filas main."""
    print("\nResumen de distribución (dataset principal):")
    total = sum(filas_por_split.values())
    for split in SPLITS_PRINCIPALES:
        filas = filas_por_split[split]
        tasa = positivos_por_split[split] / filas
        print(f"  {split:>6}: {filas:>10,} filas ({filas/total:.2%})  {tasa:.2%} congestionados")
    return total

In [18]:
cerrar_writers_previos_si_existen()
borrar_salidas_previas()
writers_x, writers_y, filas_por_split, positivos_por_split = inicializar_acumuladores()

print(f"Guardando en {DATA_FINAL.resolve()} (modo incremental por mes):")
for parquet_mes in parquets_main:
    procesar_mes_para_splits(
        parquet_mes, writers_x, writers_y, filas_por_split, positivos_por_split
    )

cerrar_writers(writers_x, writers_y)
guardar_eval_final()
total_filas_main = imprimir_resumen_distribucion(filas_por_split, positivos_por_split)

Guardando en C:\Users\gabri\Mi unidad\Proyectos Clase\analisis-datos-proyecto-final\data\final (modo incremental por mes):
[01-2026] filas=3,445,452
[02-2026] filas=2,880,568

Resumen de distribución (dataset principal):
   train:  5,060,814 filas (80.00%)  6.92% congestionados
     val:    632,603 filas (10.00%)  6.92% congestionados
    test:    632,603 filas (10.00%)  6.92% congestionados


Vemos que los archivos generados son correctos y ya están separados, así que podemos empezar el análisis en el notebook 03.